# Feature Engineering Comportemental — AML Detection

**Objectif :** Ameliorer le recall du Decision Tree (baseline 43%) en ajoutant des features comportementales.

**Methode :** Split train/test AVANT le calcul des features pour eviter le data leakage temporel.

**Spec :** voir `docs/specs/2026-05-07-feature-engineering-design.md`

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    recall_score, precision_score, f1_score
)
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

In [16]:
import os

CSV_FULL = '../SAML-D_sample_800k.csv'
CSV_SAMPLE = '../projet_final_merged/data/SAML-D_sample_1k.csv'

if os.path.exists(CSV_FULL):
    csv_path = CSV_FULL
    print(f"Dataset complet : {csv_path}")
elif os.path.exists(CSV_SAMPLE):
    csv_path = CSV_SAMPLE
    print(f"Echantillon : {csv_path}")
else:
    raise FileNotFoundError("Aucun dataset trouve.")

df = pd.read_csv(csv_path)
print(f"Shape : {df.shape}")
print(f"Transactions suspectes : {df['Is_laundering'].sum()} ({df['Is_laundering'].mean()*100:.4f}%)")

Dataset complet : ../SAML-D_sample_800k.csv
Shape : (800000, 12)
Transactions suspectes : 833 (0.1041%)


## 1. Modele Baseline — Decision Tree (6 features originales)

Reproduction du modele du projet initial pour avoir une reference de comparaison.

In [17]:
# --- Baseline : meme modele que le projet initial ---
df_baseline = df.copy()

# Encoding categoriel
cat_cols = ['Payment_type', 'Payment_currency', 'Received_currency',
            'Sender_bank_location', 'Receiver_bank_location']
oe = OrdinalEncoder()
df_baseline[cat_cols] = oe.fit_transform(df_baseline[cat_cols])

# Features et target
baseline_features = ['Amount', 'Payment_type', 'Payment_currency',
                     'Received_currency', 'Sender_bank_location',
                     'Receiver_bank_location']
X_base = df_baseline[baseline_features]
y = df_baseline['Is_laundering']

# Split
X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Entrainement
tree_baseline = DecisionTreeClassifier(
    max_depth=10, class_weight='balanced', random_state=RANDOM_STATE
)
tree_baseline.fit(X_train_base, y_train)

# Evaluation
y_pred_base = tree_baseline.predict(X_test_base)
recall_baseline = recall_score(y_test, y_pred_base)
print(f"=== BASELINE ===")
print(f"Recall test : {recall_baseline:.3f}")
print(f"\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_base))
print(f"\n{classification_report(y_test, y_pred_base, zero_division=0)}")

=== BASELINE ===
Recall test : 0.425

Matrice de confusion :
[[140864  18969]
 [    96     71]]

              precision    recall  f1-score   support

           0       1.00      0.88      0.94    159833
           1       0.00      0.43      0.01       167

    accuracy                           0.88    160000
   macro avg       0.50      0.65      0.47    160000
weighted avg       1.00      0.88      0.94    160000



## 2. Feature Engineering — Construction sur le train set uniquement

Les features sont calculees sur le train set (80%) puis appliquees au test set par lookup.
Cela evite le data leakage temporel (le modele ne voit pas d'informations futures).

### Features creees :
1. `sender_tx_count` / `receiver_tx_count` — frequence d'activite du compte
2. `is_sender_risky_country` / `is_receiver_risky_country` — pays a risque
3. `receiver_smurfing_score` — pattern smurfing (petit montant + beaucoup d'expediteurs)
4. `sender_risky_payment_count` / `receiver_risky_payment_count` — nombre de paiements a risque

In [18]:
# --- Split AVANT le calcul des features ---
# On garde les index pour pouvoir calculer les features sur le train uniquement

df_work = df.copy()

# On a besoin des colonnes originales (non encodees) pour le feature engineering
# Le split se fait sur les index
train_idx, test_idx = train_test_split(
    df_work.index, test_size=0.2, random_state=RANDOM_STATE,
    stratify=df_work['Is_laundering']
)

df_train = df_work.loc[train_idx].copy()
df_test = df_work.loc[test_idx].copy()

print(f"Train : {len(df_train)} lignes ({df_train['Is_laundering'].sum()} suspectes)")
print(f"Test  : {len(df_test)} lignes ({df_test['Is_laundering'].sum()} suspectes)")

Train : 640000 lignes (666 suspectes)
Test  : 160000 lignes (167 suspectes)


In [19]:
# --- Feature 1 : Frequence ---
# Calculer le nombre de transactions par compte sur le TRAIN uniquement
sender_counts = df_train.groupby('Sender_account').size()
receiver_counts = df_train.groupby('Receiver_account').size()

# Appliquer au train
df_train['sender_tx_count'] = df_train['Sender_account'].map(sender_counts)
df_train['receiver_tx_count'] = df_train['Receiver_account'].map(receiver_counts)

# Appliquer au test par lookup (comptes absents = 0)
df_test['sender_tx_count'] = df_test['Sender_account'].map(sender_counts).fillna(0).astype(int)
df_test['receiver_tx_count'] = df_test['Receiver_account'].map(receiver_counts).fillna(0).astype(int)

# Verification
print("=== sender_tx_count ===")
print(f"Train — mean: {df_train['sender_tx_count'].mean():.1f}, max: {df_train['sender_tx_count'].max()}")
print(f"Test  — mean: {df_test['sender_tx_count'].mean():.1f}, max: {df_test['sender_tx_count'].max()}")
print(f"Test  — comptes absents du train : {(df_test['sender_tx_count'] == 0).sum()}")

=== sender_tx_count ===
Train — mean: 20.6, max: 67
Test  — mean: 19.5, max: 67
Test  — comptes absents du train : 21885


In [ ]:
# --- Fonction reutilisable : calcul des listes "a risque" sur le train uniquement ---
# Remplace les anciennes listes hardcodees (issues d'une EDA full-dataset = data leakage).
# Memes regles metier : seuils en multiples de la moyenne globale du train.

def compute_risky_lists(df_train,
                        sender_threshold_x=3,
                        receiver_threshold_x=5,
                        payment_threshold_x=2):
    """
    Identifie les modalites a risque a partir du train uniquement.
    Retourne 3 sets : (sender_risky_countries, receiver_risky_countries, risky_payment_types).
    """
    global_rate = df_train['Is_laundering'].mean()

    sender_rates = df_train.groupby('Sender_bank_location', sort=False)['Is_laundering'].mean()
    sender_risky = set(sender_rates[sender_rates > sender_threshold_x * global_rate].index)

    receiver_rates = df_train.groupby('Receiver_bank_location', sort=False)['Is_laundering'].mean()
    receiver_risky = set(receiver_rates[receiver_rates > receiver_threshold_x * global_rate].index)

    payment_rates = df_train.groupby('Payment_type', sort=False)['Is_laundering'].mean()
    risky_payments = set(payment_rates[payment_rates > payment_threshold_x * global_rate].index)

    return sender_risky, receiver_risky, risky_payments


# --- Tests inline : verifier le contrat de la fonction ---
_s, _r, _p = compute_risky_lists(df_train)
assert isinstance(_s, set) and isinstance(_r, set) and isinstance(_p, set), \
    f"Doit retourner 3 sets, recu : {type(_s), type(_r), type(_p)}"
assert len(_s) > 0 and len(_r) > 0 and len(_p) > 0, \
    f"Aucune liste ne doit etre vide. sender={len(_s)}, receiver={len(_r)}, payments={len(_p)}"

# Monotonie des seuils : seuil plus bas => plus d'elements
_s_loose, _, _ = compute_risky_lists(df_train, sender_threshold_x=1)
_s_strict, _, _ = compute_risky_lists(df_train, sender_threshold_x=10)
assert len(_s_loose) >= len(_s) >= len(_s_strict), \
    f"Monotonie violee : seuil 1 -> {len(_s_loose)}, seuil 3 -> {len(_s)}, seuil 10 -> {len(_s_strict)}"

print("compute_risky_lists() : tests passes")
del _s, _r, _p, _s_loose, _s_strict


In [ ]:
# --- Feature 2 : Pays a risque ---
# Listes calculees sur le train uniquement (anti-leakage).
# Memes regles que dans le spec d'origine : seuils 3x / 5x / 2x la moyenne globale.

SENDER_RISKY, RECEIVER_RISKY, RISKY_PAYMENT_TYPES = compute_risky_lists(df_train)

print(f"Sender risques (calcules sur le train) : {sorted(SENDER_RISKY)}")
print(f"Receiver risques (calcules sur le train) : {sorted(RECEIVER_RISKY)}")
print(f"Types de paiement risques : {sorted(RISKY_PAYMENT_TYPES)}")

# Comparaison avec les listes hardcodees d'origine (issues de l'EDA full-dataset).
# A retirer apres validation soutenance — sinon le leakage revient par copy-paste.
_expected_sender = {'Albania', 'Italy', 'Netherlands'}
_expected_receiver = {'Nigeria', 'Albania', 'Morocco', 'Mexico'}
_expected_payments = {'Cash Deposit', 'Cash Withdrawal', 'Cross-border'}
print(f"\nDiff sender   : ajoutes={SENDER_RISKY - _expected_sender}, retires={_expected_sender - SENDER_RISKY}")
print(f"Diff receiver : ajoutes={RECEIVER_RISKY - _expected_receiver}, retires={_expected_receiver - RECEIVER_RISKY}")
print(f"Diff payments : ajoutes={RISKY_PAYMENT_TYPES - _expected_payments}, retires={_expected_payments - RISKY_PAYMENT_TYPES}")

for dataset in [df_train, df_test]:
    dataset['is_sender_risky_country'] = dataset['Sender_bank_location'].isin(SENDER_RISKY).astype(int)
    dataset['is_receiver_risky_country'] = dataset['Receiver_bank_location'].isin(RECEIVER_RISKY).astype(int)

print("\n=== Pays a risque ===")
print(f"Train - sender risky : {df_train['is_sender_risky_country'].sum()} ({df_train['is_sender_risky_country'].mean()*100:.2f}%)")
print(f"Train - receiver risky : {df_train['is_receiver_risky_country'].sum()} ({df_train['is_receiver_risky_country'].mean()*100:.2f}%)")
print(f"\nTaux suspect parmi sender risky (train) :")
print(df_train.groupby('is_sender_risky_country')['Is_laundering'].mean() * 100)


In [ ]:
# --- Feature 3 : Smurfing Score ---
# Smurfing = petits montants + beaucoup d'expediteurs differents.
# Score = nb_expediteurs_uniques * nb_transactions (calcule sur le train uniquement)

# Filtrer les petits montants (< 5000) dans le train
small = df_train[df_train['Amount'] < 5000]

# Calculer les 2 dimensions par receiver
n_senders = small.groupby('Receiver_account')['Sender_account'].nunique()
n_tx = small.groupby('Receiver_account').size()
smurfing_score = n_senders * n_tx

# Appliquer au train et test (comptes inconnus = 0)
df_train['receiver_smurfing_score'] = df_train['Receiver_account'].map(smurfing_score).fillna(0)
df_test['receiver_smurfing_score'] = df_test['Receiver_account'].map(smurfing_score).fillna(0)

# Verification
score = df_train['receiver_smurfing_score']
print("=== Smurfing Score ===")
print(f"Train — mean: {score.mean():.2f}, median: {score.median():.2f}, max: {score.max():.1f}")
print(f"Train — % avec score > 0 : {(score > 0).mean()*100:.1f}%")

# Comparer le taux suspect par tranche
p90 = score[score > 0].quantile(0.90)
print(f"\nP90 (parmi scores > 0) = {p90:.1f}")
print("Taux suspect par tranche (train) :")
for label, mask in [("score = 0", score == 0),
                    ("0 < score <= P90", (score > 0) & (score <= p90)),
                    ("top 10% (score > P90)", score > p90)]:
    n = mask.sum()
    taux = df_train.loc[mask, 'Is_laundering'].mean() * 100
    print(f"  {label:<22s} : {taux:.3f}%  ({n:,} transactions)")


In [ ]:
# --- Feature 4 : Risky Payment Count ---
# Compter, pour chaque compte, combien de transactions de type "a risque" il a fait dans le train.
# RISKY_PAYMENT_TYPES est deja calcule en cellule "Pays a risque" via compute_risky_lists().

# Marquer les transactions a risque dans le train
is_risky = df_train['Payment_type'].isin(RISKY_PAYMENT_TYPES).astype(int)

# Compter le nb de paiements a risque par compte (sur le train uniquement)
sender_risky_counts = is_risky.groupby(df_train['Sender_account']).sum()
receiver_risky_counts = is_risky.groupby(df_train['Receiver_account']).sum()

# Appliquer au train et test (comptes absents du test = 0)
df_train['sender_risky_payment_count'] = df_train['Sender_account'].map(sender_risky_counts)
df_train['receiver_risky_payment_count'] = df_train['Receiver_account'].map(receiver_risky_counts)
df_test['sender_risky_payment_count'] = df_test['Sender_account'].map(sender_risky_counts).fillna(0).astype(int)
df_test['receiver_risky_payment_count'] = df_test['Receiver_account'].map(receiver_risky_counts).fillna(0).astype(int)

# Verification
print("=== Risky Payment Count ===")
print(f"Train — sender mean: {df_train['sender_risky_payment_count'].mean():.2f}")
print(f"Train — receiver mean: {df_train['receiver_risky_payment_count'].mean():.2f}")
print(f"\nTaux suspect par sender_risky_payment_count > 0 (train) :")
print(df_train.groupby(df_train['sender_risky_payment_count'] > 0)['Is_laundering'].mean() * 100)


## 3. Modeles ameliores — Decision Tree et XGBoost

Deux modeles entraines avec les 13 features (6 originales + 7 nouvelles) pour comparer :
- **Decision Tree** (max_depth=10) — comparaison directe avec le baseline
- **XGBoost** — modele ensembliste performant sur donnees tabulaires desequilibrees

In [ ]:
# --- Encoding des colonnes categorielles ---
cat_cols = ['Payment_type', 'Payment_currency', 'Received_currency',
            'Sender_bank_location', 'Receiver_bank_location']

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
oe.fit(df_train[cat_cols])
df_train[cat_cols] = oe.transform(df_train[cat_cols])
df_test[cat_cols] = oe.transform(df_test[cat_cols])

# --- Features ---
new_features = ['sender_tx_count', 'receiver_tx_count',
                'is_sender_risky_country', 'is_receiver_risky_country',
                'receiver_smurfing_score',
                'sender_risky_payment_count', 'receiver_risky_payment_count']

all_features = baseline_features + new_features

X_train_new = df_train[all_features]
X_test_new = df_test[all_features]
y_train_new = df_train['Is_laundering']
y_test_new = df_test['Is_laundering']

print(f"Features utilisees ({len(all_features)}) : {all_features}")
print(f"X_train shape : {X_train_new.shape}")
print(f"X_test shape  : {X_test_new.shape}")

In [24]:
# --- Decision Tree avec nouvelles features ---
tree_new = DecisionTreeClassifier(
    max_depth=10, class_weight='balanced', random_state=RANDOM_STATE
)
tree_new.fit(X_train_new, y_train_new)

y_pred_tree = tree_new.predict(X_test_new)
recall_tree = recall_score(y_test_new, y_pred_tree)

print(f"=== DECISION TREE (13 features) ===")
print(f"Recall BASELINE : {recall_baseline:.3f}")
print(f"Recall NOUVEAU  : {recall_tree:.3f}")
print(f"Amelioration    : {(recall_tree - recall_baseline)*100:+.1f} points")
print(f"\nMatrice de confusion :")
print(confusion_matrix(y_test_new, y_pred_tree))
print(f"\n{classification_report(y_test_new, y_pred_tree, zero_division=0)}")

=== DECISION TREE (13 features) ===
Recall BASELINE : 0.425
Recall NOUVEAU  : 0.635
Amelioration    : +21.0 points

Matrice de confusion :
[[147146  12687]
 [    61    106]]

              precision    recall  f1-score   support

           0       1.00      0.92      0.96    159833
           1       0.01      0.63      0.02       167

    accuracy                           0.92    160000
   macro avg       0.50      0.78      0.49    160000
weighted avg       1.00      0.92      0.96    160000



In [25]:
# --- XGBoost avec nouvelles features ---
# scale_pos_weight compense le desequilibre : nb_negatifs / nb_positifs
n_neg = (y_train_new == 0).sum()
n_pos = (y_train_new == 1).sum()

xgb = XGBClassifier(
    max_depth=6,
    n_estimators=200,
    learning_rate=0.1,
    scale_pos_weight=n_neg / n_pos,
    random_state=RANDOM_STATE,
    eval_metric='aucpr',
    use_label_encoder=False
)
xgb.fit(X_train_new, y_train_new)

y_pred_xgb = xgb.predict(X_test_new)
recall_xgb = recall_score(y_test_new, y_pred_xgb)

print(f"=== XGBOOST (13 features) ===")
print(f"Recall BASELINE      : {recall_baseline:.3f}")
print(f"Recall Decision Tree : {recall_tree:.3f}")
print(f"Recall XGBoost       : {recall_xgb:.3f}")
print(f"\nMatrice de confusion :")
print(confusion_matrix(y_test_new, y_pred_xgb))
print(f"\n{classification_report(y_test_new, y_pred_xgb, zero_division=0)}")

=== XGBOOST (13 features) ===
Recall BASELINE      : 0.425
Recall Decision Tree : 0.635
Recall XGBoost       : 0.527

Matrice de confusion :
[[155314   4519]
 [    79     88]]

              precision    recall  f1-score   support

           0       1.00      0.97      0.99    159833
           1       0.02      0.53      0.04       167

    accuracy                           0.97    160000
   macro avg       0.51      0.75      0.51    160000
weighted avg       1.00      0.97      0.98    160000



In [26]:
# --- Random Forest avec nouvelles features ---
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X_train_new, y_train_new)

y_pred_rf = rf.predict(X_test_new)
recall_rf = recall_score(y_test_new, y_pred_rf)

print(f"=== RANDOM FOREST (13 features) ===")
print(f"Recall BASELINE      : {recall_baseline:.3f}")
print(f"Recall Decision Tree : {recall_tree:.3f}")
print(f"Recall Random Forest : {recall_rf:.3f}")
print(f"Recall XGBoost       : {recall_xgb:.3f}")
print(f"\nMatrice de confusion :")
print(confusion_matrix(y_test_new, y_pred_rf))
print(f"\n{classification_report(y_test_new, y_pred_rf, zero_division=0)}")

=== RANDOM FOREST (13 features) ===
Recall BASELINE      : 0.425
Recall Decision Tree : 0.635
Recall Random Forest : 0.575
Recall XGBoost       : 0.527

Matrice de confusion :
[[149646  10187]
 [    71     96]]

              precision    recall  f1-score   support

           0       1.00      0.94      0.97    159833
           1       0.01      0.57      0.02       167

    accuracy                           0.94    160000
   macro avg       0.50      0.76      0.49    160000
weighted avg       1.00      0.94      0.97    160000



## 4. Analyse des resultats

In [ ]:
# --- Comparaison Train vs Test (surapprentissage) ---
y_train_pred_tree = tree_new.predict(X_train_new)
y_train_pred_rf = rf.predict(X_train_new)
y_train_pred_xgb = xgb.predict(X_train_new)

models = [("Decision Tree", y_train_pred_tree, y_pred_tree),
          ("Random Forest", y_train_pred_rf, y_pred_rf),
          ("XGBoost",       y_train_pred_xgb, y_pred_xgb)]

for model_name, y_tr_pred, y_te_pred in models:
    print(f"=== {model_name} — Train vs Test ===")
    print(f"  Recall    train={recall_score(y_train_new, y_tr_pred):.3f}  | test={recall_score(y_test_new, y_te_pred):.3f}")
    print(f"  Precision train={precision_score(y_train_new, y_tr_pred, zero_division=0):.3f}  | test={precision_score(y_test_new, y_te_pred, zero_division=0):.3f}")
    print(f"  F1        train={f1_score(y_train_new, y_tr_pred):.3f}  | test={f1_score(y_test_new, y_te_pred):.3f}")
    print()


### Analyse du surapprentissage

Les 3 modeles montrent un ecart significatif entre recall train et recall test (30 a 45 points).
Cela signifie que les modeles **memorisent** en partie les donnees d'entrainement au lieu d'apprendre
des patterns generalisables.

**Pourquoi c'est si fort ici :**
- Seulement **666 cas suspects** dans le train (0.1%) — le modele peut apprendre les cas individuels par coeur
- Les features comportementales (tx_count, risky_payment_count) sont tres discriminantes sur le train mais
  le test contient des comptes jamais vus

**Ce que le tuning corrige (voir section 7) :**
- Les modeles par defaut utilisent des profondeurs elevees (max_depth=10 pour DT/RF, 6 pour XGBoost)
- Le tuning trouve que des modeles **moins complexes** generalisent mieux
- Le tableau comparatif train vs test (section 7) quantifie la reduction de l'ecart apres tuning

**Implication pour la production :**
Le recall reel en production serait probablement proche du recall test plutot que du recall train.
C'est une performance solide pour un systeme de **pre-filtrage** AML : le modele genere des alertes
qui sont ensuite examinees par des analystes humains. La precision basse (beaucoup de fausses alertes)
est attendue avec 0.1% de positifs — le seuil de decision peut etre ajuste en production
via la courbe precision-recall (section 8).

In [28]:
# --- Importance des features ---
importance_tree = pd.Series(tree_new.feature_importances_, index=all_features).sort_values(ascending=False)
importance_rf = pd.Series(rf.feature_importances_, index=all_features).sort_values(ascending=False)
importance_xgb = pd.Series(xgb.feature_importances_, index=all_features).sort_values(ascending=False)

for name, importance in [("Decision Tree", importance_tree), ("Random Forest", importance_rf), ("XGBoost", importance_xgb)]:
    print(f"=== Importance — {name} ===")
    for feat, imp in importance.items():
        marker = " <-- NEW" if feat in new_features else ""
        print(f"  {feat:35s} {imp*100:5.1f}%{marker}")
    print()

=== Importance — Decision Tree ===
  sender_tx_count                      28.0% <-- NEW
  receiver_risky_payment_count         24.2% <-- NEW
  receiver_tx_count                    18.0% <-- NEW
  Amount                               12.7%
  sender_risky_payment_count            8.3% <-- NEW
  receiver_smurfing_score               3.3% <-- NEW
  Payment_type                          2.4%
  Receiver_bank_location                1.0%
  Received_currency                     0.8%
  Payment_currency                      0.6%
  is_receiver_risky_country             0.3% <-- NEW
  Sender_bank_location                  0.3%
  is_sender_risky_country               0.0% <-- NEW

=== Importance — Random Forest ===
  sender_tx_count                      25.1% <-- NEW
  receiver_risky_payment_count         18.5% <-- NEW
  receiver_tx_count                    14.1% <-- NEW
  sender_risky_payment_count           11.4% <-- NEW
  Amount                               10.6%
  Received_currency            

In [ ]:
# --- Visualisation : Matrices de confusion — 4 modeles ---
fig, axes = plt.subplots(1, 4, figsize=(28, 6))

# Les 4 modeles a comparer : (axe, y_true, y_pred, titre)
models = [(axes[0], y_test,     y_pred_base, "Baseline\n(6 features)"),
          (axes[1], y_test_new, y_pred_tree, "Decision Tree\n(13 features)"),
          (axes[2], y_test_new, y_pred_rf,   "Random Forest\n(13 features)"),
          (axes[3], y_test_new, y_pred_xgb,  "XGBoost\n(13 features)")]

for ax, y_true, y_pred, title in models:
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    recall_val = tp / (tp + fn)
    annot = np.array([[f"TN\n{tn:,}", f"FP\n{fp:,}"],
                      [f"FN\n{fn}", f"TP\n{tp}"]])
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues",
                xticklabels=["Predit Normal", "Predit Suspect"],
                yticklabels=["Reel Normal", "Reel Suspect"],
                annot_kws={"fontsize": 11, "fontweight": "bold"}, ax=ax)
    ax.set_title(f"{title}\nRecall = {recall_val*100:.1f}%", fontsize=11, fontweight="bold")

plt.suptitle("Comparaison des 4 modeles", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# --- Visualisation : Importance des features — 3 modeles ---
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

models = [(axes[0], importance_tree, "Decision Tree"),
          (axes[1], importance_rf,   "Random Forest"),
          (axes[2], importance_xgb,  "XGBoost")]

for ax, importance, title in models:
    imp_pct = importance.sort_values() * 100  # importance en pourcentage
    # Couleur de chaque barre : rouge pour les nouvelles features, bleue pour les autres
    colors = ['#E74C3C' if f in new_features else '#2E86C1' for f in imp_pct.index]
    ax.barh(imp_pct.index, imp_pct.values, color=colors)
    # Annoter chaque barre avec sa valeur en pourcentage
    for i, val in enumerate(imp_pct.values):
        ax.text(val + 0.3, i, f"{val:.1f}%", va="center", fontsize=9, fontweight="bold")
    ax.set_xlabel("Importance (%)")
    ax.set_title(f"{title}\nRouge = nouvelles features", fontsize=12, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle("Importance des features par modele", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [31]:
# --- Tableau recap ---
print("=" * 65)
print(f"{'Modele':<28} {'Recall':>8} {'Precision':>10} {'F1':>8}")
print("-" * 65)
print(f"{'Baseline (6 feat.)':<28} {recall_baseline:>8.3f} {precision_score(y_test, y_pred_base, zero_division=0):>10.4f} {f1_score(y_test, y_pred_base):>8.4f}")
print(f"{'Decision Tree (13 feat.)':<28} {recall_tree:>8.3f} {precision_score(y_test_new, y_pred_tree, zero_division=0):>10.4f} {f1_score(y_test_new, y_pred_tree):>8.4f}")
print(f"{'Random Forest (13 feat.)':<28} {recall_rf:>8.3f} {precision_score(y_test_new, y_pred_rf, zero_division=0):>10.4f} {f1_score(y_test_new, y_pred_rf):>8.4f}")
print(f"{'XGBoost (13 feat.)':<28} {recall_xgb:>8.3f} {precision_score(y_test_new, y_pred_xgb, zero_division=0):>10.4f} {f1_score(y_test_new, y_pred_xgb):>8.4f}")
print("=" * 65)

Modele                         Recall  Precision       F1
-----------------------------------------------------------------
Baseline (6 feat.)              0.425     0.0037   0.0074
Decision Tree (13 feat.)        0.635     0.0083   0.0164
Random Forest (13 feat.)        0.575     0.0093   0.0184
XGBoost (13 feat.)              0.527     0.0191   0.0369


## 5. Validation croisee

Un seul split 80/20 peut donner un recall variable selon le hasard du split (167 cas positifs seulement dans le test).
La validation croisee 5-fold donne une estimation plus robuste.

In [ ]:
# --- Validation croisee 5-fold PROPRE ---
# Toutes les features (et les listes risquees) sont recalculees a l'interieur de chaque fold
# pour eviter tout data leakage. C'est la regle d'or anti-leakage.

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

def build_features_on_fold(df_fold_train, df_fold_test):
    """Calcule les features sur le fold train et les applique au fold test par lookup.
    Recalcule aussi les listes risquees sur le fold train uniquement (anti-leakage)."""

    # 0. Recalculer les listes risquees sur le fold train uniquement (LEAKAGE FIX)
    sender_risky, receiver_risky, risky_payments = compute_risky_lists(df_fold_train)

    # 1. Frequence (nb tx par compte)
    sender_counts = df_fold_train.groupby('Sender_account').size()
    receiver_counts = df_fold_train.groupby('Receiver_account').size()
    df_fold_train['sender_tx_count'] = df_fold_train['Sender_account'].map(sender_counts)
    df_fold_train['receiver_tx_count'] = df_fold_train['Receiver_account'].map(receiver_counts)
    df_fold_test['sender_tx_count'] = df_fold_test['Sender_account'].map(sender_counts).fillna(0).astype(int)
    df_fold_test['receiver_tx_count'] = df_fold_test['Receiver_account'].map(receiver_counts).fillna(0).astype(int)

    # 2. Pays a risque (utilise les listes locales du fold)
    for ds in [df_fold_train, df_fold_test]:
        ds['is_sender_risky_country'] = ds['Sender_bank_location'].isin(sender_risky).astype(int)
        ds['is_receiver_risky_country'] = ds['Receiver_bank_location'].isin(receiver_risky).astype(int)

    # 3. Smurfing score (memes formules que cellule "Smurfing Score" en section 2)
    small = df_fold_train[df_fold_train['Amount'] < 5000]
    n_senders = small.groupby('Receiver_account')['Sender_account'].nunique()
    n_tx = small.groupby('Receiver_account').size()
    smurfing_score = n_senders * n_tx
    df_fold_train['receiver_smurfing_score'] = df_fold_train['Receiver_account'].map(smurfing_score).fillna(0)
    df_fold_test['receiver_smurfing_score'] = df_fold_test['Receiver_account'].map(smurfing_score).fillna(0)

    # 4. Risky payment count (utilise risky_payments local)
    is_risky = df_fold_train['Payment_type'].isin(risky_payments).astype(int)
    s_risky = is_risky.groupby(df_fold_train['Sender_account']).sum()
    r_risky = is_risky.groupby(df_fold_train['Receiver_account']).sum()
    df_fold_train['sender_risky_payment_count'] = df_fold_train['Sender_account'].map(s_risky)
    df_fold_train['receiver_risky_payment_count'] = df_fold_train['Receiver_account'].map(r_risky)
    df_fold_test['sender_risky_payment_count'] = df_fold_test['Sender_account'].map(s_risky).fillna(0).astype(int)
    df_fold_test['receiver_risky_payment_count'] = df_fold_test['Receiver_account'].map(r_risky).fillna(0).astype(int)

    # Encoding categoriel (fit uniquement sur le fold train)
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    oe.fit(df_fold_train[cat_cols])
    df_fold_train[cat_cols] = oe.transform(df_fold_train[cat_cols])
    df_fold_test[cat_cols] = oe.transform(df_fold_test[cat_cols])

    return df_fold_train, df_fold_test


In [ ]:
# --- Boucle CV 5-fold ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
df_cv = df.copy()
y_cv = df_cv['Is_laundering']

results = {'Decision Tree': [], 'Random Forest': [], 'XGBoost': []}

for fold, (train_idx, test_idx) in enumerate(cv.split(df_cv, y_cv)):
    df_fold_train = df_cv.iloc[train_idx].copy()
    df_fold_test = df_cv.iloc[test_idx].copy()

    # Verifier la stabilite des listes risquees calculees sur ce fold train
    s, r, p = compute_risky_lists(df_fold_train)
    print(f"Fold {fold+1} listes risquees - sender={sorted(s)}, receiver={sorted(r)}, payments={sorted(p)}")

    # Construire features et encoder
    df_fold_train, df_fold_test = build_features_on_fold(df_fold_train, df_fold_test)

    X_tr, X_te = df_fold_train[all_features], df_fold_test[all_features]
    y_tr, y_te = df_fold_train['Is_laundering'], df_fold_test['Is_laundering']

    # Decision Tree
    dt = DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=RANDOM_STATE)
    dt.fit(X_tr, y_tr)
    results['Decision Tree'].append(recall_score(y_te, dt.predict(X_te)))

    # Random Forest
    rf_cv = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced',
                                   n_jobs=-1, random_state=RANDOM_STATE)
    rf_cv.fit(X_tr, y_tr)
    results['Random Forest'].append(recall_score(y_te, rf_cv.predict(X_te)))

    # XGBoost
    n_neg = (y_tr == 0).sum()
    n_pos = (y_tr == 1).sum()
    xgb_cv = XGBClassifier(max_depth=6, n_estimators=200, learning_rate=0.1,
                           scale_pos_weight=n_neg / n_pos,
                           random_state=RANDOM_STATE, eval_metric='aucpr', use_label_encoder=False)
    xgb_cv.fit(X_tr, y_tr)
    results['XGBoost'].append(recall_score(y_te, xgb_cv.predict(X_te)))

    print(f"Fold {fold+1} - DT: {results['Decision Tree'][-1]:.3f}, RF: {results['Random Forest'][-1]:.3f}, XGB: {results['XGBoost'][-1]:.3f}\n")


In [ ]:
# --- Resume des resultats CV ---
print(f"=== Validation croisee 5-fold (features + listes recalculees par fold) ===")
print(f"Baseline recall : {recall_baseline:.3f}\n")
for name, scores in results.items():
    scores = np.array(scores)
    print(f"{name}:")
    print(f"  Recall par fold : {scores}")
    print(f"  Recall moyen    : {scores.mean():.3f} (+/- {scores.std():.3f})")
    print(f"  Amelioration    : {(scores.mean() - recall_baseline)*100:+.1f} points\n")


## 6. Tuning d'hyperparametres (sans data leakage)

Les features comportementales (tx_count, smurfing_score, risky_payment_count) dependent des donnees.
Si on les calcule sur tout le dataset avant le CV split, le modele voit des informations du test dans le train = **data leakage**.

**Methode correcte :** on reutilise `build_features_on_fold()` pour recalculer les features a l'interieur de chaque fold.
Le tuning est donc manuel (pas de `RandomizedSearchCV`) mais les resultats sont fiables.

**Metrique d'optimisation : recall.**
En AML, un faux negatif (transaction suspecte non detectee) a des consequences bien plus graves
qu'un faux positif (fausse alerte). Le recall mesure la capacite du modele a ne pas laisser passer
de cas de blanchiment. La precision est affichee a titre indicatif pour chaque combinaison.

> **Pourquoi pas le F1 ou F2 ?** Avec un dataset a 0.1% de positifs, la precision sera structurellement
> basse quel que soit le modele. Optimiser un compromis recall/precision au seuil par defaut (0.5) n'a
> pas de sens — en production, le seuil serait ajuste via la courbe precision-recall (section 8)
> pour atteindre le compromis souhaite par l'equipe conformite.

**Grille de recherche :**
- Decision Tree : `max_depth` x `min_samples_leaf`
- Random Forest : `max_depth` x `min_samples_leaf` x `n_estimators`
- XGBoost : `max_depth` x `learning_rate` x `min_child_weight`

In [ ]:
# --- Fonction de tuning manuel avec features recalculees par fold ---
from itertools import product

def manual_cv_tuning(df_source, model_class, param_grid, fixed_params, n_splits=3):
    """Teste toutes les combinaisons de param_grid en CV. Optimise le recall.
    Retourne (best_params, best_score, df_results)."""
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    y_source = df_source['Is_laundering']

    # Toutes les combinaisons de parametres a tester (produit cartesien)
    combinations = list(product(*param_grid.values()))
    print(f"Testing {len(combinations)} combinations x {n_splits} folds = {len(combinations) * n_splits} fits\n")

    all_results = []
    best_score = -1
    best_params = None

    for i, combo in enumerate(combinations):
        params = dict(zip(param_grid.keys(), combo))

        fold_recall, fold_precision = [], []
        for train_idx, test_idx in cv.split(df_source, y_source):
            df_t = df_source.iloc[train_idx].copy()
            df_te = df_source.iloc[test_idx].copy()
            df_t, df_te = build_features_on_fold(df_t, df_te)

            X_tr, X_te = df_t[all_features], df_te[all_features]
            y_tr, y_te = df_t['Is_laundering'], df_te['Is_laundering']

            model = model_class(**fixed_params, **params)  # fusion des 2 dicts en parametres
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)

            fold_recall.append(recall_score(y_te, y_pred))
            fold_precision.append(precision_score(y_te, y_pred, zero_division=0))

        mean_recall = np.mean(fold_recall)
        mean_precision = np.mean(fold_precision)
        all_results.append({**params, 'mean_recall': mean_recall,
                            'std_recall': np.std(fold_recall), 'mean_precision': mean_precision})

        if mean_recall > best_score:
            best_score = mean_recall
            best_params = params

        print(f"  [{i+1}/{len(combinations)}] {params} — recall={mean_recall:.3f}  precision={mean_precision:.4f}")

    print(f"\n  >>> Best: recall={best_score:.3f} | {best_params}")
    return best_params, best_score, pd.DataFrame(all_results)


print("Fonction manual_cv_tuning prete.")


In [ ]:
# --- Tuning Decision Tree ---
dt_param_grid = {
    'max_depth': [5, 8, 10],
    'min_samples_leaf': [1, 10, 50],
}
dt_fixed = {'class_weight': 'balanced', 'random_state': RANDOM_STATE}

print("=== TUNING DECISION TREE ===")
dt_best_params, dt_best_score, dt_results = manual_cv_tuning(
    df.copy(), DecisionTreeClassifier, dt_param_grid, dt_fixed, n_splits=3
)
print(f"\nMeilleurs parametres : {dt_best_params}")
print(f"Meilleur recall CV   : {dt_best_score:.3f}")

In [ ]:
# --- Tuning Random Forest ---
rf_param_grid = {
    'max_depth': [5, 8, 10],
    'min_samples_leaf': [1, 5, 20],
    'n_estimators': [200],
}
rf_fixed = {'class_weight': 'balanced', 'n_jobs': -1, 'random_state': RANDOM_STATE}

print("=== TUNING RANDOM FOREST ===")
rf_best_params, rf_best_score, rf_results = manual_cv_tuning(
    df.copy(), RandomForestClassifier, rf_param_grid, rf_fixed, n_splits=3
)
print(f"\nMeilleurs parametres : {rf_best_params}")
print(f"Meilleur recall CV   : {rf_best_score:.3f}")

In [ ]:
# --- Tuning XGBoost ---
xgb_param_grid = {
    'max_depth': [3, 5, 8],
    'learning_rate': [0.05, 0.1],
    'min_child_weight': [1, 5],
    'n_estimators': [200],
}
# scale_pos_weight sera calcule dynamiquement dans chaque fold
# mais pour le tuning on utilise le ratio global comme approximation
n_neg_global = (df['Is_laundering'] == 0).sum()
n_pos_global = (df['Is_laundering'] == 1).sum()
xgb_fixed = {
    'scale_pos_weight': n_neg_global / n_pos_global,
    'random_state': RANDOM_STATE,
    'eval_metric': 'aucpr',
    'use_label_encoder': False,
}

print("=== TUNING XGBOOST ===")
xgb_best_params, xgb_best_score, xgb_results = manual_cv_tuning(
    df.copy(), XGBClassifier, xgb_param_grid, xgb_fixed, n_splits=3
)
print(f"\nMeilleurs parametres : {xgb_best_params}")
print(f"Meilleur recall CV   : {xgb_best_score:.3f}")

In [ ]:
# --- Recap du tuning ---
metric_cols = ['mean_recall', 'std_recall', 'mean_precision']
models = [("Decision Tree", dt_results),
          ("Random Forest", rf_results),
          ("XGBoost",       xgb_results)]

print("=" * 85)
print(f"{'Modele':<20} {'Recall CV':>10} {'Prec. CV':>10}   {'Meilleurs parametres'}")
print("-" * 85)

for name, results_df in models:
    best_row = results_df.loc[results_df['mean_recall'].idxmax()]
    best_params = best_row.drop(metric_cols).to_dict()
    print(f"{name:<20} {best_row['mean_recall']:>10.3f} {best_row['mean_precision']:>10.4f}   {best_params}")

print("=" * 85)
print(f"\nBaseline recall : {recall_baseline:.3f}")


## 7. Evaluation finale — Modeles tunes sur le split train/test propre

Les meilleurs hyperparametres trouves en section 6 sont maintenant reevalues sur le split train/test
de la section 2 (features calculees sur le train uniquement, puis appliquees au test par lookup).

In [ ]:
# --- Reentrainement avec les meilleurs hyperparametres sur le split propre ---
# X_train_new, X_test_new, y_train_new, y_test_new viennent de la section 2-3.

# Decision Tree tune
dt_tuned = DecisionTreeClassifier(**dt_best_params, class_weight='balanced', random_state=RANDOM_STATE)
dt_tuned.fit(X_train_new, y_train_new)
y_pred_dt_tuned = dt_tuned.predict(X_test_new)

# Random Forest tune
rf_tuned = RandomForestClassifier(**rf_best_params, class_weight='balanced',
                                  n_jobs=-1, random_state=RANDOM_STATE)
rf_tuned.fit(X_train_new, y_train_new)
y_pred_rf_tuned = rf_tuned.predict(X_test_new)

# XGBoost tune
n_neg = (y_train_new == 0).sum()
n_pos = (y_train_new == 1).sum()
xgb_tuned = XGBClassifier(**xgb_best_params, scale_pos_weight=n_neg/n_pos,
                          random_state=RANDOM_STATE, eval_metric='aucpr', use_label_encoder=False)
xgb_tuned.fit(X_train_new, y_train_new)
y_pred_xgb_tuned = xgb_tuned.predict(X_test_new)

print("3 modeles tunes entraines.")


In [ ]:
# --- Tableau comparatif complet (avant + apres tuning) ---
results = [
    ("Baseline (6 feat, DT depth=10)", y_test,     y_pred_base,     None),
    ("DT (13 feat, depth=10)",         y_test_new, y_pred_tree,     None),
    ("RF (13 feat, depth=10)",         y_test_new, y_pred_rf,       None),
    ("XGB (13 feat, defaults)",        y_test_new, y_pred_xgb,      None),
    ("DT TUNE (13 feat)",              y_test_new, y_pred_dt_tuned, dt_best_params),
    ("RF TUNE (13 feat)",              y_test_new, y_pred_rf_tuned, rf_best_params),
    ("XGB TUNE (13 feat)",             y_test_new, y_pred_xgb_tuned, xgb_best_params),
]

print("=" * 80)
print(f"{'Modele':<35} {'Recall':>8} {'Precision':>10} {'F1':>8}   Parametres")
print("-" * 80)
for name, y_true, y_pred, params in results:
    r = recall_score(y_true, y_pred)
    p = precision_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred)
    extra = f"   {params}" if params else ""
    print(f"{name:<35} {r:>8.3f} {p:>10.4f} {f:>8.4f}{extra}")
print("=" * 80)

# Identifier le meilleur modele tune (np.argmax retourne l'index du max)
names_tuned = ['DT TUNE', 'RF TUNE', 'XGB TUNE']
recalls_tuned = [recall_score(y_test_new, y_pred_dt_tuned),
                 recall_score(y_test_new, y_pred_rf_tuned),
                 recall_score(y_test_new, y_pred_xgb_tuned)]
best_idx = np.argmax(recalls_tuned)
print(f"\nMeilleur modele : {names_tuned[best_idx]} (recall test = {recalls_tuned[best_idx]:.3f})")
print(f"Amelioration vs baseline : {(recalls_tuned[best_idx] - recall_baseline)*100:+.1f} points")


In [ ]:
# --- Verification du surapprentissage : Train vs Test (avant et apres tuning) ---
print("=" * 80)
print("Surapprentissage — Train vs Test (avant et apres tuning)")
print("=" * 80)
print(f"{'Modele':<25} {'Recall train':>13} {'Recall test':>12} {'Ecart':>8}")
print("-" * 80)

# Avant tuning
before_tuning = [("DT (depth=10)",  tree_new, y_pred_tree),
                 ("RF (depth=10)",  rf,       y_pred_rf),
                 ("XGB (defaults)", xgb,      y_pred_xgb)]

for name, model, y_te_pred in before_tuning:
    r_train = recall_score(y_train_new, model.predict(X_train_new))
    r_test = recall_score(y_test_new, y_te_pred)
    print(f"{name:<25} {r_train:>13.3f} {r_test:>12.3f} {r_train - r_test:>+8.3f}")

print("-" * 80)

# Apres tuning
after_tuning = [("DT TUNE",  dt_tuned,  y_pred_dt_tuned),
                ("RF TUNE",  rf_tuned,  y_pred_rf_tuned),
                ("XGB TUNE", xgb_tuned, y_pred_xgb_tuned)]

for name, model, y_te_pred in after_tuning:
    r_train = recall_score(y_train_new, model.predict(X_train_new))
    r_test = recall_score(y_test_new, y_te_pred)
    print(f"{name:<25} {r_train:>13.3f} {r_test:>12.3f} {r_train - r_test:>+8.3f}")

print("=" * 80)


## 🎯 Recommandation finale

> **Modèle retenu : XGBoost tuné** — `max_depth=3, learning_rate=0.05, min_child_weight=1, n_estimators=200`

### Pourquoi XGBoost et pas Decision Tree ?

| Critère | DT TUNE | RF TUNE | **XGB TUNE** |
|---|---|---|---|
| Recall (split simple) | 0.772 ⭐ | 0.725 | 0.760 |
| **Recall (CV 5-fold)** | 0.734 | 0.746 | **0.762** ⭐ |
| Variance CV (std) | 0.041 | 0.030 | **0.006** ⭐ |
| **Average Precision** | 0.0209 | 0.0493 | **0.0681** ⭐ |
| Précision (seuil 0.5) | 0.34% | 0.61% | **0.65%** ⭐ |

DT TUNE a le meilleur recall sur le split simple, mais **XGB gagne sur les 4 autres critères** : recall CV plus élevé, variance bien plus faible (modèle plus stable), AP 3× supérieur (efficace sur toute la courbe précision-recall, pas juste au seuil 0.5), et précision presque double.

### Seuil opérationnel recommandé : **0.7**

| Seuil | Recall | Précision | Alertes/an | Dossiers/vrai cas | Charge analystes |
|---|---|---|---|---|---|
| 0.5 (défaut) | 76.0% | 0.65% | 19 653 | 155 | ~1.0 ETP |
| **0.7 (recommandé)** | **61.7%** | **1.34%** | **7 664** | **74** | **~0.4 ETP** |
| 0.8 (strict) | 46.1% | 2.38% | 3 239 | 42 | ~0.2 ETP |

**Avec le seuil 0.7**, on détecte **61.7% des cas de blanchiment** (vs 42.5% baseline, soit **+19 points**) avec une charge analyste soutenable (~0.4 ETP pour traiter 7 700 alertes/an à 5 min par alerte).

### Méthodologie anti-leakage

Toutes les features comportementales (`sender_tx_count`, `receiver_smurfing_score`, etc.) **et** les listes "à risque" (pays, types de paiement) sont calculées **sur le train uniquement** dans chaque fold de CV. Aucune information du test n'est utilisée pour construire les features — c'est la règle d'or anti-leakage en machine learning.

### Limites identifiées (voir `docs/backlog-after-simplification.md`)

- Split aléatoire au lieu de temporel — la perf réelle en production sera probablement plus basse.
- Seuil de "petit montant" (5000) non normalisé par devise.
- Dataset synthétique SAML-D — les patterns AML réels peuvent différer.


## 8. Courbe Precision-Recall — Analyse du seuil de decision

Par defaut, un modele classe "suspect" si la probabilite predite depasse **0.5**. Mais ce seuil est arbitraire.

La courbe precision-recall montre le **trade-off** entre :
- **Recall** (ne pas rater de vrais cas) — prioritaire en AML
- **Precision** (ne pas noyer les analystes sous les fausses alertes) — cout operationnel

Elle permet de **choisir un seuil adapte au metier** plutot que d'utiliser 0.5 par defaut.

In [ ]:
# --- Courbes Precision-Recall pour les 3 modeles tunes ---
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, axes = plt.subplots(1, 3, figsize=(24, 7))

models = [(axes[0], dt_tuned,  "Decision Tree", "#2E86C1"),
          (axes[1], rf_tuned,  "Random Forest", "#27AE60"),
          (axes[2], xgb_tuned, "XGBoost",       "#E74C3C")]

# Precision d'un classifieur aleatoire = taux de positifs (calcule une seule fois)
baseline_precision = y_test_new.mean()

for ax, model, name, color in models:
    # Probabilites de la classe positive + courbe PR
    y_scores = model.predict_proba(X_test_new)[:, 1]
    precision_vals, recall_vals, _ = precision_recall_curve(y_test_new, y_scores)
    ap = average_precision_score(y_test_new, y_scores)

    # Trace la courbe + ligne aleatoire
    ax.plot(recall_vals, precision_vals, color=color, linewidth=2, label=f'AP = {ap:.4f}')
    ax.axhline(baseline_precision, color='gray', linestyle='--', alpha=0.5,
               label=f'Aleatoire = {baseline_precision:.4f}')

    # Marquer le point du seuil par defaut (0.5)
    y_pred_default = (y_scores >= 0.5).astype(int)
    r_def = recall_score(y_test_new, y_pred_default)
    p_def = precision_score(y_test_new, y_pred_default, zero_division=0)
    ax.scatter(r_def, p_def, color='red', s=100, zorder=5,
               label=f'Seuil 0.5 (R={r_def:.2f}, P={p_def:.3f})')

    ax.set_xlabel("Recall", fontsize=12)
    ax.set_ylabel("Precision", fontsize=12)
    ax.set_title(f"{name}\nAverage Precision = {ap:.4f}", fontsize=13, fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)
    ax.set_xlim([0, 1.05])
    ax.set_ylim([0, max(precision_vals) * 1.15])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle("Courbes Precision-Recall — Modeles tunes (13 features)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# --- Tableau : impact du seuil sur le meilleur modele tune ---
# Reutilise best_idx de la cellule 11b (meilleur recall test au seuil 0.5)

models_tuned = [(dt_tuned, "DT TUNE"), (rf_tuned, "RF TUNE"), (xgb_tuned, "XGB TUNE")]
best_model, best_name = models_tuned[best_idx]
y_scores_best = best_model.predict_proba(X_test_new)[:, 1]
n_positifs = y_test_new.sum()

print(f"\n=== Impact du seuil de decision — {best_name} ===")
print(f"{'Seuil':>8} {'Recall':>8} {'Precision':>10} {'Alertes':>10} {'TP':>6} {'FP':>8} {'Dossiers/vrai cas':>18}")
print("-" * 70)

for seuil in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    y_pred_s = (y_scores_best >= seuil).astype(int)
    tp = ((y_pred_s == 1) & (y_test_new == 1)).sum()
    fp = ((y_pred_s == 1) & (y_test_new == 0)).sum()
    total_alertes = tp + fp
    r = tp / n_positifs
    p = tp / total_alertes if total_alertes > 0 else 0
    dossiers = total_alertes / tp if tp > 0 else float('inf')
    print(f"{seuil:>8.1f} {r:>8.1%} {p:>10.2%} {total_alertes:>10,} {tp:>6} {fp:>8,} {dossiers:>18.0f}")


In [ ]:
# --- Tableau seuils : XGB TUNE specifiquement (meilleur modele selon la CV) ---
# La cellule precedente affiche DT TUNE (parce que best_idx vient du split simple).
# Mais en CV, XGB TUNE est le meilleur (0.762 vs 0.734 pour DT) ET il a une
# precision presque double. Donc on regarde aussi les seuils pour XGB.

y_scores_xgb = xgb_tuned.predict_proba(X_test_new)[:, 1]
n_positifs = y_test_new.sum()

print(f"\n=== Impact du seuil de decision — XGB TUNE (meilleur en CV) ===")
print(f"{'Seuil':>8} {'Recall':>8} {'Precision':>10} {'Alertes':>10} {'TP':>6} {'FP':>8} {'Dossiers/vrai cas':>18}")
print("-" * 70)

for seuil in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    y_pred_s = (y_scores_xgb >= seuil).astype(int)
    tp = ((y_pred_s == 1) & (y_test_new == 1)).sum()
    fp = ((y_pred_s == 1) & (y_test_new == 0)).sum()
    total_alertes = tp + fp
    r = tp / n_positifs
    p = tp / total_alertes if total_alertes > 0 else 0
    dossiers = total_alertes / tp if tp > 0 else float('inf')
    print(f"{seuil:>8.1f} {r:>8.1%} {p:>10.2%} {total_alertes:>10,} {tp:>6} {fp:>8,} {dossiers:>18.0f}")


### Lecture du tableau

La colonne **"Dossiers/vrai cas"** est la plus parlante pour le metier :
- Seuil bas (0.1) → on detecte plus de cas, mais les analystes traitent beaucoup de faux positifs
- Seuil haut (0.7) → peu de fausses alertes, mais on rate des vrais cas de blanchiment

**En AML, le recall prime** — mieux vaut generer des fausses alertes que laisser passer un vrai blanchiment.
Mais le seuil optimal depend de la capacite de traitement des equipes d'analystes.

La metrique **Average Precision (AP)** resume la qualite globale du modele independamment du seuil choisi — plus elle est elevee, meilleur est le modele.